In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/drive/MyDrive/AIML/trumptweets_small.csv', encoding="ISO-8859-1")
df.head()

,id,link,content,date,retweets,favorites,mentions,hashtags,geo
0,1698308935,https://twitter.com/realDonaldTrump/status/169...,Be sure to tune in and watch Donald Trump on L...,2009-05-04 20:54:25,500,868,NaN,NaN,NaN
1,1701461182,https://twitter.com/realDonaldTrump/status/170...,Donald Trump will be appearing on The View tom...,2009-05-05 03:00:10,33,273,NaN,NaN,NaN
2,1737479987,https://twitter.com/realDonaldTrump/status/173...,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08 15:38:08,12,18,NaN,NaN,NaN
3,1741160716,https://twitter.com/realDonaldTrump/status/174...,New Blog Post: Celebrity Apprentice Finale and...,2009-05-08 22:40:15,11,24,NaN,NaN,NaN
4,1773561338,https://twitter.com/realDonaldTrump/status/177...,"""My persona will never be that of a wallflower...",2009-05-12 16:07:28,1399,1965,NaN,NaN,NaN


In [4]:
data_text = df["content"].dropna()

In [5]:
data_text[11]

'"We win in our lives by having a champion\'s view of each moment." --Donald J. Trump http://tinyurl.com/pqpfvm'

**Cleaning Functions**

In [6]:
import re
def remove_urls(text):
  """
  This function will try to remove URL present in out dataset and replace it with space using regex library.
  Input Args:
  text: strings of text that may contain URLs.
  Output Args:
  text: URLs replaces with text
  """
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)


In [7]:
def remove_emoji(string):
  """
  This function will replace the emoji in string with whitespace
  """
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)

In [8]:
def removeunwanted_characters(document):
  """
  This function will remove all the unwanted characters from the input dataset.
  Input Args:
  documet: A text data to be cleaned.
  Return:
  A cleaned document.
  """
  # remove user mentions
  document = re.sub("@[A-Za-z0-9_]+"," ", document)
  # remove hashtags
  document = re.sub("#[A-Za-z0-9_]+","", document)
  # remove punctuation
  document = re.sub("[^0-9A-Za-z ]", "" , document)
  #remove emojis
  document = remove_emoji(document)
  # remove double spaces
  document = document.replace('  ',"")
  return document.strip()

In [9]:
def lower_order(text):
  """
  This function converts all the text in input text to lower order.
  Input Args:
  token_text : input text.
  Returns:
  small_order_text : text converted to small/lower order.
  """
  small_order_text = text.lower()
  return small_order_text

# Test:
sample_text = "This Is some Normalized TEXT"
sample_small = lower_order(sample_text)
print(sample_small)


this is some normalized text


**Token Generation**

In [10]:
import nltk
nltk.download('punkt')
from nltk import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [11]:
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize,pos_tag
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

def lemmatization(token_text):
  """
  This function performs the lemmatization operations as explained above.
  Input Args:
  token_text: list of tokens.
  Returns:
  lemmatized_tokens: list of lemmatized tokens.
  """
  lemma_tokens = []
  wordnet = WordNetLemmatizer()
  lemmatized_tokens = [wordnet.lemmatize(token, pos = 'v') for token in token_text]

  return lemmatized_tokens




[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [12]:
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))
custom_stopwords = ['@', 'RT']
stop_words.update(custom_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [13]:

def remove_stopwords(text_tokens):
  """
  This function removes all the stopwords present in out text tokens.
  Input Args:
  text_tokens: tokenize input of our datasets.
  Returns:
  result_tokens: list of token without stopword.
  """

  result_tokens = []
  for token in text_tokens:
    if token not in stop_words:
       result_tokens.append(token)
  return result_tokens

In [14]:
from nltk.stem import PorterStemmer

def stemming(text):
  """
  This function performs stemming operations.
  Input Args:
  token_text: list of tokenize text.
  Returns:
  stemm_tokes: list of stemmed tokens.
  """
  porter = PorterStemmer()
  stemm_tokens = []
  for word in text:
    stemm_tokens.append(porter.stem(word))
  return stemm_tokens

In [15]:
from nltk.tokenize import RegexpTokenizer

from nltk.tokenize import RegexpTokenizer

def remove_punct(text):
  """
  This function removes the punctutations present in our text data.
  Input Args:
  text: text data.
  Returns:
  text: cleaned text.
  """
  tokenizer = RegexpTokenizer(r"\w+")
  lst=tokenizer.tokenize(' '.join(text))
  return lst


In [16]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    This function applies all text cleaning steps to the input dataset.
    Input Args:
      dataset: A string of text to be cleaned.
      rule: "lemmatize" or "stem" — controls the normalization method.
    Returns:
      A cleaned string of text.
    """
    # Convert the input to small/lower order
    data = lower_order(dataset)

    # Remove URLs
    data = remove_urls(data)

    # Remove emojis
    data = remove_emoji(data)

    # Remove all other unwanted characters
    data = removeunwanted_characters(data)

    # Create tokens
    tokens = data.split()

    # Remove stopwords
    tokens = remove_stopwords(tokens)

    if rule == "lemmatize":
        tokens = lemmatization(tokens)
    elif rule == "stem":
        tokens = stemming(tokens)
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

In [17]:
test = data_text[0]

In [18]:
print(text_cleaning_pipeline(test))

sure tune watch donald trump late night david letterman present top ten list tonight


In [22]:
cleaned_text = data_text.apply(lambda dataset: text_cleaning_pipeline(dataset))


In [20]:
cleaned_text.head(20)

,content
0,sure tune watch donald trump late night david ...
1,donald trump appear view tomorrow morning disc...
2,donald trump read top ten financial tip late s...
3,new blog post celebrity apprentice finale less...
4,persona never wallflowerid rather build wall c...
5,miss usa tara conner firedive always believer ...
6,listen interview donald trump discuss new book...
7,strive wholeness keep sense wonder intact dona...
8,enter think like champion sign book keychain c...
9,achiever achieve plateau begin donald j trump


In [26]:
type(cleaned_text)

pandas.core.series.Series

In [28]:
tokens = cleaned_text.apply(lambda x: x.split())

In [31]:
tokens = cleaned_text.apply(str.split)
tokens.head(20)

,content
0,"[sure, tune, watch, donald, trump, late, night..."
1,"[donald, trump, appear, view, tomorrow, mornin..."
2,"[donald, trump, read, top, ten, financial, tip..."
3,"[new, blog, post, celebrity, apprentice, final..."
4,"[persona, never, wallflowerid, rather, build, ..."
5,"[miss, usa, tara, conner, firedive, always, be..."
6,"[listen, interview, donald, trump, discuss, ne..."
7,"[strive, wholeness, keep, sense, wonder, intac..."
8,"[enter, think, like, champion, sign, book, key..."
9,"[achiever, achieve, plateau, begin, donald, j,..."


In [33]:
tokens_no_stop = tokens.apply(remove_stopwords)

In [34]:
tokens_lemma = tokens_no_stop.apply(lemmatization)

**TF-IDF**

In [35]:
final_text = tokens_lemma.apply(lambda x: " ".join(x))

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(final_text)

In [37]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(final_text)

sequences = tokenizer.texts_to_sequences(final_text)

In [39]:
print(sequences[:5])

[[244, 421, 50, 14, 3, 679, 88, 619, 1187, 983, 231, 1892, 722, 79], [14, 3, 1745, 504, 163, 215, 233, 290, 149, 13, 147, 37, 17, 404], [14, 3, 166, 231, 1892, 930, 1854, 679, 44, 619, 1187, 972], [13, 2676, 388, 290, 149, 1893, 2507, 393, 518, 64], [4587, 30, 1, 680, 100, 146, 1, 14, 531, 3]]


In [40]:
print(tokenizer.word_index)

{'<OOV>': 1, 'great': 2, 'trump': 3, 'thank': 4, 'get': 5, 'make': 6, 'president': 7, 'go': 8, 'people': 9, 'us': 10, 'would': 11, 'time': 12, 'new': 13, 'donald': 14, 'say': 15, 'country': 16, 'like': 17, 'run': 18, 'america': 19, 'job': 20, 'good': 21, 'need': 22, 'big': 23, 'look': 24, 'want': 25, 'one': 26, 'dont': 27, 'vote': 28, 'obama': 29, 'never': 30, 'work': 31, 'love': 32, 'many': 33, 'know': 34, 'state': 35, 'see': 36, 'think': 37, 'realdonaldtrump': 38, 'much': 39, 'take': 40, 'back': 41, 'win': 42, 'give': 43, 'show': 44, 'news': 45, 'come': 46, 'best': 47, 'even': 48, 'deal': 49, 'watch': 50, 'today': 51, 'must': 52, 'democrats': 53, 'really': 54, 'border': 55, 'last': 56, 'years': 57, 'interview': 58, 'true': 59, 'cant': 60, 'bad': 61, 'hillary': 62, 'china': 63, 'way': 64, 'american': 65, 'ever': 66, 'day': 67, 'keep': 68, 'right': 69, 'fake': 70, 'world': 71, 'first': 72, 'poll': 73, 'call': 74, 'better': 75, 'media': 76, 'nothing': 77, 'im': 78, 'tonight': 79, 'unite

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(
    sequences,
    maxlen=50,
    padding='post',
    truncating='post'
)

print(X.shape)

(41122, 50)
